# Testing ADAPT.jl interface in python

In [1]:
import subprocess
import os
from datetime import datetime
from pathlib import Path
import json
from itertools import islice 
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from src.adapt_utils import (
    run_adapt_jl_parallel,
    show_adapt_logs,
    get_combined_res_df
)

In [4]:
adapt_gpt_dir = Path(
    "/home/mrzaizai2k/code_bao/ADAPT_GPT"
)
adapt_output_dir = "./ADAPT.jl_results/qaoa_python_test"
n_graphs = 5
n_nodes = 10
input_graph_filename = "ADAPT.jl_results/graphs_n10.json"


In [5]:
import networkx as nx
import random

def add_weights_to_nx_graph(G, weighted=True, use_negative=False):
    elist = []
    for u, v in G.edges():
        if weighted:
            w = random.uniform(0.1, 1.0)
            if use_negative and random.random() < 0.5:
                w *= -1
        else:
            w = -1 if (use_negative and random.random() < 0.5) else 1

        elist.append([int(u)+1, int(v)+1, float(round(w, 2))])  
        # +1 to match Julia 1-indexing
    return elist

In [6]:
def generate_graphs(
    n_graphs=10,
    n_nodes=10,
    density=None,          # if None → random
    weighted=True,
    use_negative=False
):
    graphs_dict = {}

    for i in range(n_graphs):
        if density is None:
            p = random.uniform(0.6, 0.9)   # random density
        else:
            p = density

        G = nx.erdos_renyi_graph(n=n_nodes, p=p)

        # avoid empty graph
        while G.number_of_edges() == 0:
            G = nx.erdos_renyi_graph(n=n_nodes, p=p)

        elist = add_weights_to_nx_graph(G, weighted, use_negative)

        graph_name = f"Graph_{i}_n{n_nodes}"

        graphs_dict[graph_name] = {
            "elist": elist,
            "n_nodes": n_nodes
        }

    return graphs_dict

In [7]:
import json

def save_graphs_to_json(graphs_dict, filename):
    with open(filename, "w") as f:
        json.dump(graphs_dict, f, indent=2)

In [8]:
def load_graphs(filename):
    with open(filename, "r") as f:
        return json.load(f)

In [9]:


graphs = generate_graphs(
    n_graphs=n_graphs,
    n_nodes=n_nodes,
    density=None,          # or e.g. 0.7
    weighted=True,
    use_negative=False
)

save_graphs_to_json(graphs, input_graph_filename)

# Load back
cur_input_graphs_dict = load_graphs(input_graph_filename)

# Access like your image
print(cur_input_graphs_dict["Graph_0_n10"])

{'elist': [[1, 3, 0.45], [1, 4, 0.69], [1, 5, 0.1], [1, 6, 0.95], [1, 7, 0.79], [1, 8, 0.19], [1, 9, 0.16], [1, 10, 0.38], [2, 3, 0.68], [2, 4, 0.66], [2, 5, 0.6], [2, 6, 0.56], [2, 7, 0.73], [2, 10, 0.24], [3, 4, 0.85], [3, 5, 1.0], [3, 6, 0.42], [3, 8, 0.57], [3, 9, 0.37], [3, 10, 0.72], [4, 8, 0.93], [4, 9, 0.52], [4, 10, 0.12], [5, 6, 0.69], [5, 7, 0.77], [5, 8, 0.83], [5, 9, 0.23], [5, 10, 0.8], [6, 7, 0.21], [6, 8, 0.59], [6, 9, 0.7], [6, 10, 0.65], [7, 9, 0.21], [7, 10, 0.25], [8, 9, 0.9], [8, 10, 0.5], [9, 10, 0.2]], 'n_nodes': 10}


In [ ]:
logs_list, cur_proc = run_adapt_jl_parallel(
    script_dir=adapt_gpt_dir,
    output_dir=adapt_output_dir,
    input_graphs=adapt_gpt_dir / input_graph_filename,
    n_workers=2,
    graphs_number=n_graphs,
    trials_per_graph=1,
    max_params=50,
    gamma_0="gamma0_grid.json",
    pool_name="qaoa_double_pool",
    use_floor_stopper=True,
    temp_folder="temp_data",
)

Splitting input graphs into 2 parts
Starting worker 0 on node: DESKTOP-H2CRQMR
Starting worker 1 on node: DESKTOP-H2CRQMR


In [14]:
show_adapt_logs(logs_list, n_lines=20)

Log: worker_DESKTOP-H2CRQMR_0
┌────────┬─────────────┬──────────┐
│  iter  │    value    │   time   │
├────────┼─────────────┼──────────┤
│      1 │       0.000 │     2.00 │
│        │     965.000 │     4.00 │
│        │      99.000 │     1.00 │
└────────┴─────────────┴──────────┘

Number of callbacks: 6
Exact energy (MQLib): -13.98
Running with optimizer: BFGS
For QAOA reading gamma_0 values from: gamma0_grid.json
For QAOA using gamma_0 = 0.01, trial: 1/1, graph N: 2/3
Success!
final energy:	-12.794999999996719 (through trace)
Took time: 1.4 sec.;
N layers: 7;
g0 = 0.01;
ar = 0.9152360515019112
For QAOA using gamma_0 = 0.1, trial: 1/1, graph N: 2/3

--------------------------------------------------
Log: worker_DESKTOP-H2CRQMR_1

Graph name: Graph_3_n10;
Number of edges: 35;
Number of nodes: 10;
Generator: input_file.

▷ MQLib
▷ Heuristic: BURER2002
┌────────┬─────────────┬──────────┐
│  iter  │    value    │   time   │
├────────┼─────────────┼──────────┤
│      1 │       0.000 │     

In [15]:
show_adapt_logs(logs_list, n_lines=20, pbar_only=True)

Log: worker_DESKTOP-H2CRQMR_0
Graphs on: unknown; pid: 71587: 33.3%┣███▊       ┫ 1/3 [00:26<Inf:Inf, InfGs/it]

Log: worker_DESKTOP-H2CRQMR_1
Graphs on: unknown; pid: 71589: 50.0%┣█████▌     ┫ 1/2 [00:23<Inf:Inf, InfGs/it]



In [16]:
combined_res_df = get_combined_res_df(
    adapt_output_dir,
    debug_limit=5,
)

Reading ADAPT.jl results from:
	 /home/mrzaizai2k/code_bao/ADAPT_GPT/ADAPT.jl_results/qaoa_python_test


Opening ADAPT results (qaoa_python_test): 100%|██████████| 2/2 [00:00<00:00, 239.98it/s]


df_list len: 2


Opening graphs (qaoa_python_test):   0%|          | 0/2 [00:00<?, ?it/s]

Opening graphs (qaoa_python_test): 100%|██████████| 2/2 [00:00<00:00, 739.21it/s]

df_list len: 2
Graphs count:
g_method
input_file    2
Name: count, dtype: int64
Aggregating results...


In [17]:
combined_res_df.columns

Index(['graph_num', 'run', 'prefix', 'method', 'optimizer', 'gamma0',
       'pooltype', 'graph_name', 'n_nodes', 'energy_list', 'took_time',
       'energy_mqlib', 'op_list', 'approx_ratio', 'edge_weight_norm_coef',
       'β_coeff', 'γ_coeff', 'coeff', 'energy_eigen', 'cut_mqlib', 'cut_adapt',
       'cut_eig', 'state_vect_adapt', 'success_flag', 'g_method',
       'edgelist_json', 'H_frob_norm', 'worker_id', 'edgelist_list',
       'edgelist_list_len', 'num_connected_comp', 'n_layers', 'graph_id'],
      dtype='object')

In [18]:
temp_df = combined_res_df.drop(columns=["prefix", "energy_list", "coeff", "cut_mqlib", "cut_adapt", "cut_eig", "state_vect_adapt", "graph_id"])

In [19]:
temp_df[:1]['approx_ratio']

0    0.978169
Name: approx_ratio, dtype: float64

In [21]:
temp_df[:1]['energy_eigen']

0   -12.83
Name: energy_eigen, dtype: float64

In [22]:
temp_df[:1]['energy_mqlib']

0   -12.83
Name: energy_mqlib, dtype: float64

In [20]:
temp_df[:1]

,graph_num,run,method,optimizer,gamma0,pooltype,graph_name,n_nodes,took_time,energy_mqlib,op_list,approx_ratio,edge_weight_norm_coef,β_coeff,γ_coeff,energy_eigen,success_flag,g_method,edgelist_json,H_frob_norm,worker_id,edgelist_list,edgelist_list_len,num_connected_comp,n_layers
0,1,1,qaoa,BFGS,0.01,qaoa_double_pool,Graph_2_n10,10,8.362022,-12.83,"[171, 62, 75, 126, 106, 150, 42, 169, 30, 20, 162, 119, 122, 41, 1, 126]",0.978169,1.0,"[0.7857033043391813, 0.785856613563815, -32.201453969251524, 0.7863168599339343, 22.77654887923753, -0.7854378740087693, 0.7528415899022993, -0.7854090466972611, -0.0360818995966899, 0.000364002422288, 0.0006479228910043, 0.0005861444489849, 0.7839446338085361, 0.0005954571864754, -0.0032876995990097, 0.7850959724323372]","[-0.0011866382731537, 3.954713963649374e-05, 0.0029681319024348, 0.003701254260115, -0.0048981482549863, -0.0024533534652603, -0.1164930200322466, 1.7199695634644885, 0.0776965557227414, 0.0774634125114953, 0.0776724935753277, 0.0763383736654927, 0.0759789288554924, 0.0011123351942032, 0.000706737288295, 0.0006422288614066]",-12.83,True,input_file,"[[1,2,0.52],[1,3,0.38],[2,3,0.29],[1,4,0.41],[2,4,0.52],[3,4,0.16],[1,5,0.18],[2,5,0.28],[3,5,0.54],[4,5,0.21],[1,6,0.11],[2,6,0.87],[3,6,0.64],[4,6,0.16],[5,6,0.43],[2,7,0.58],[3,7,0.89],[4,7,0.95],[5,7,0.39],[6,7,0.41],[1,8,0.17],[2,8,0.22],[3,8,0.37],[4,8,0.29],[5,8,0.48],[6,8,0.8],[7,8,0.95],[1,9,0.28],[3,9,0.24],[4,9,0.76],[5,9,0.54],[6,9,0.77],[7,9,0.32],[8,9,0.6],[1,10,0.23],[2,10,0.16],[3,10,0.64],[4,10,0.56],[7,10,0.36],[8,10,0.73]]",298.87803,worker_unknown_pid_71587_ts_26-04-05__14_57_graphs_json,"[[1, 2, 0.52], [1, 3, 0.38], [2, 3, 0.29], [1, 4, 0.41], [2, 4, 0.52], [3, 4, 0.16], [1, 5, 0.18], [2, 5, 0.28], [3, 5, 0.54], [4, 5, 0.21], [1, 6, 0.11], [2, 6, 0.87], [3, 6, 0.64], [4, 6, 0.16], [5, 6, 0.43], [2, 7, 0.58], [3, 7, 0.89], [4, 7, 0.95], [5, 7, 0.39], [6, 7, 0.41], [1, 8, 0.17], [2, 8, 0.22], [3, 8, 0.37], [4, 8, 0.29], [5, 8, 0.48], [6, 8, 0.8], [7, 8, 0.95], [1, 9, 0.28], [3, 9, 0.24], [4, 9, 0.76], [5, 9, 0.54], [6, 9, 0.77], [7, 9, 0.32], [8, 9, 0.6], [1, 10, 0.23], [2, 10, 0.16], [3, 10, 0.64], [4, 10, 0.56], [7, 10, 0.36], [8, 10, 0.73]]",40,1,16


We can read graph from edgelist_json, use our model to create data and comapre with approx_ratio, compare n_layers